# Markov chain represented by a finite state-machine

## Introduction

### Markov Chain

Sequence of possible events in which the probability of each event depends only on the state attained in the previous event.
The Markov property can be formulated as $P(X_{n}=x_{n}\mid X_{n-1}=x_{n-1},...,X_{0}=x_{0})=P(X_{n}=x_{n}\mid X_{n-1}=x_{n-1})$\

The strong Markov property depends on a starting (i.e., current) state (i.e., present) which resembles a random variable called the stopping time. See the examples of https://en.wikipedia.org/wiki/Stopping_time.

A state which cannot be reobtained after leaving is called a 'transient state'.\
A state which has a probability to be revisited of 1 is called a 'recurrent state'.\
A Markov chain is irreducible if every state can be obtained in x steps starting from any state.
If there are states which, once obtained, do not lead to some or any other state, the Markov chain is reducible.

When considering a stationary distribution, one should resolve the chain into classes and consider all irreducible classes, without taking the transient ones into account.

If there is a state which is ultimately an end of the Markov chain, the chain will be 'absorbed' into this state. If there are several possible absorbing states then there are certain probabilities leading to one or another. The Markov Chain then has several possible stationary states (non-unique, see https://towardsdatascience.com/the-gamblers-ruin-problem-9c97a7747171).

See https://en.wikipedia.org/wiki/Markov_chain#Properties for more info.

The destination is the column of the matrix, the start is the row.

### Finite state machine

Mathematical model of computation - the machine can be in exactly one of a finite number of states at any given time.

## Simple example code

In [1]:
import numpy as np

In [9]:
rng = np.random.default_rng(100)

In [75]:
transition_matrix_reducible = np.array([[1, 0, 0], [0.1, 0.5, 0.4], [0.2, 0.2, 0.6]])
transition_matrix_reducible

array([[1. , 0. , 0. ],
       [0.1, 0.5, 0.4],
       [0.2, 0.2, 0.6]])

In [17]:
transition_matrix_irreducible = np.array([[0, 0.4, 0.6], [0.1, 0.5, 0.4], [0.2, 0.2, 0.6]])
transition_matrix_irreducible

array([[0. , 0.4, 0.6],
       [0.1, 0.5, 0.4],
       [0.2, 0.2, 0.6]])

In [7]:
initial_state = 1

In [11]:
list_of_states = []
current_state = initial_state
for i in range(10):
    current_state = rng.choice([0, 1, 2], p=transition_matrix_irreducible[current_state])
    list_of_states.append(current_state)
list_of_states

[2, 2, 1, 0, 2, 2, 2, 2, 2, 0]

### Monte Carlo to derive equilibrium distribution

The equilibrium distribution denotes the expected probabilities for the different states.

In [12]:
n_simulations = int(1e5)
pi = np.array([0, 0, 0])
pi[initial_state] += 1

for i in range(n_simulations):
    current_state = rng.choice([0, 1, 2], p=transition_matrix_irreducible[current_state])
    pi[current_state] += 1
pi / (n_simulations+1)

array([0.14062859, 0.32477675, 0.53459465])

### Repeated matrix multiplication

Probability of reaching state 1 from state 0 after 1 step is e.g. 0.4.

Probability of reaching state j from state i after exactly n steps?

https://www.youtube.com/watch?v=Zo3ieESzr4E&list=PLM8wYQRetTxBkdvBtz-gw8b9lcVkdXQKV&index=3

$P_{ij}(n)=A^{n}_{ij}$ where $A$ is the transition matrix

also see Chapman-Kolmogorov Theorem, Master equation

Probability of reaching state j after an infinite number of transitions? Doesn't depend on the starting state.
Conditions for **unique** stationary distribution $\pi$: irreducibility and aperiodicity

$\lim_{n \to \infty} A^{n}$

Converges quicker to the stationary distribution than Monte Carlo

In [13]:
n_simulations = int(1e3)
transition_matrix_2 = transition_matrix_irreducible

for i in range(n_simulations):
    transition_matrix_2 = np.matmul(transition_matrix_2, transition_matrix_irreducible)
    
transition_matrix_2

array([[0.13953488, 0.3255814 , 0.53488372],
       [0.13953488, 0.3255814 , 0.53488372],
       [0.13953488, 0.3255814 , 0.53488372]])

### Eigenvalue approach
see https://brilliant.org/wiki/stationary-distributions/

In [16]:
import sympy as sy

In [18]:
transposed = transition_matrix_irreducible.T

Note: numpy and scipy may have problems with floating point precision leading to complex numbers or false signs.

In [24]:
m = sy.Matrix(transposed)
m.eigenvects()

[(-0.156155281280883,
  1,
  [Matrix([
   [ 0.814442028234794],
   [-0.357089816409817],
   [-0.457352211824977]])]),
 (0.256155281280883,
  1,
  [Matrix([
   [ 0.171931216155796],
   [-0.784273322665113],
   [ 0.612342106509317]])]),
 (1.00000000000000,
  1,
  [Matrix([
   [0.241681508032867],
   [0.563923518743355],
   [0.926445780792655]])])]

Pick the positive eigenvector

In [29]:
result = np.array(m.eigenvects()[2][2])
result

array([[[0.241681508032867],
        [0.563923518743355],
        [0.926445780792655]]], dtype=object)

The results may not be normalized to e.g., 1

In [30]:
result = result/(np.sum(result))
result

array([[[0.139534883720930],
        [0.325581395348837],
        [0.534883720930233]]], dtype=object)

### Chapman-Kolmogorov theorem

$P_{ij}(n)=\sum_{k}P(X_{n-r}=j\mid X_{r}=k)P(X_{r}=k\mid X_{0}=i)$\
Here, the source is i, the destination is j, the steps are n. The intermediate stop at state k is accomplished after r steps.

The Markov property refers to the memoryless property of a stochastic process and is essential for this to work.

If the probability distribution on the state space of the Markov chain is discrete and the Markov chain is homogeneous, the Chapman-Kolmogorov equation can be expressed in terms of matrix multiplication $P_{ij}(n)=\sum_{k}P_{ik}(r)\times P_{kj}(n-r)$ and one can get $P(n)=P^{n}$, which is the repeated matrix multiplication.\

See https://www.youtube.com/watch?v=Zo3ieESzr4E&list=PLM8wYQRetTxBkdvBtz-gw8b9lcVkdXQKV&index=3, \
https://de.wikipedia.org/wiki/Chapman-Kolmogorow-Gleichung, \
https://en.wikipedia.org/wiki/Chapman–Kolmogorov_equation 

### Master equation

The master equation is the differtial form of the Chapman-Kolmogorov equation.\
The matrix from is $\frac{d\vec{P}}{dt}=A\vec{P}$ where A is the matrix of connections and $\vec{P}$ is a column vector.\
Another form is $\frac{dP_{k}}{dt} =\sum_{l}A_{kl}P_{l}$ where k is the destination and l the source. 

See https://en.wikipedia.org/wiki/Master_equation